In [3]:
import os
import json
import random
os.chdir(os.path.dirname(os.getcwd()))

In [4]:
from helpers.utils import walk_dir

In [5]:
model_dir = 'data/models'

In [119]:
models   = [
            'Qwen_Qwen3-14b', 
           "kenpath_mhv_vistaar_qwen3-14b_v0.1",
            'gpt-4.1', 
    "kenpath_mhv_vistaar_qwen3-14b_v0.2",
    # 'openai_gpt-oss-20b',
    # 'openai_gpt-oss-120b',
    # 'mistralai_Mistral-Small-3.2-24B-Instruct-2506',
    # 'gpt-4.1-mini',
           ]

for model_name in models:
    model_data_fp = os.path.join(model_dir,model_name,'evaluation.json')
    data = json.load(open(model_data_fp))
    print(len(data))

470
470
470
470


In [123]:
import json
from pathlib import Path

def recalculate_overall_average(evaluation: dict) -> float | None:
    """Recalculate overall_average from all valid sub-dimension scores."""
    dimensions = evaluation.get("dimensions", {})
    all_scores = []
    
    for dim_name, dim_data in dimensions.items():
        scores_dict = dim_data.get("scores", {})
        for sub_dim_name, sub_dim_data in scores_dict.items():
            score = sub_dim_data.get("score")
            if score is not None:
                all_scores.append(score)
    
    if all_scores:
        return round(sum(all_scores) / len(all_scores), 2)
    return None

def fix_evaluation_file(filepath: Path):
    """Fix overall_average in an evaluation.json file."""
    with open(filepath, 'r', encoding='utf-8') as f:
        records = json.load(f)
    
    for record in records:
        if "evaluation" in record:
            evaluation = record["evaluation"]
            new_avg = recalculate_overall_average(evaluation)
            evaluation["metrics"]["overall_average"] = new_avg
    
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(records, f, ensure_ascii=False, indent=4)
    
    print(f"Fixed: {filepath}")

# Fix all evaluation.json files
models_dir = Path("data/models")
for eval_file in models_dir.glob("*/evaluation.json"):
    fix_evaluation_file(eval_file)

Fixed: data/models/Qwen_Qwen3-14b/evaluation.json
Fixed: data/models/openai_gpt-oss-120b/evaluation.json
Fixed: data/models/kenpath_mhv_vistaar_qwen3-14b_v0.1/evaluation.json
Fixed: data/models/gpt-4.1/evaluation.json
Fixed: data/models/gpt-4.1-mini/evaluation.json
Fixed: data/models/openai_gpt-oss-20b/evaluation.json
Fixed: data/models/mistralai_Mistral-Small-3.2-24B-Instruct-2506/evaluation.json
Fixed: data/models/kenpath_mhv_vistaar_qwen3-14b_v0.2/evaluation.json


In [94]:
record = random.choice(data)

In [95]:
dimensions = record['evaluation']['dimensions'].keys()
dimensions

dict_keys(['process_fidelity', 'factual_grounding', 'response_usefulness', 'marathi_quality'])

In [97]:
scores = [record['evaluation']['dimensions'][DIMENSION]['average'] for DIMENSION in dimensions]
print(scores)
print("Average", np.average(scores))
record['evaluation']['metrics']

[2.67, 2.0, 2.0, 2.25]
Average 2.23


{'overall_average': 2.23,
 'critical_failures': [],
 'critical_failure_count': 0,
 'overall_pass': True}

In [40]:
import numpy as np

np.float64(1.5)

In [7]:
from enum import IntEnum
from typing import Optional, Dict, Any, List


In [10]:
from pydantic import BaseModel, Field


In [11]:
class Rating(IntEnum):
    """Likert-scale rating for evaluation sub-dimensions."""
    UNACCEPTABLE = 1  # Critical failure - harmful, fabricated, or complete breakdown
    POOR = 2          # Major issues significantly affecting usefulness or correctness
    ACCEPTABLE = 3    # Meets minimum requirements with noticeable gaps
    GOOD = 4          # Strong performance with only minor issues
    EXCELLENT = 5     # Exemplary, matches gold-standard behavior


class SubDimensionScore(BaseModel):
    """Score for a single evaluation sub-dimension."""
    score: Optional[Rating] = Field(
        None, 
        description="Rating 1-5, or null if not applicable to this query type"
    )
    evidence: str = Field(
        ..., 
        description="Brief justification with reasoning and supporting snippets"
    )

# =============================================================================
# DIMENSION 1: PROCESS FIDELITY
# =============================================================================

class ProcessFidelity(BaseModel):
    """Evaluates whether the agent followed prescribed workflows and maintained output hygiene."""
    
    agristack_workflow: SubDimensionScore = Field(
        ..., 
        description="Called fetch_agristack_data first when marked available"
    )
    term_identification: SubDimensionScore = Field(
        ..., 
        description="Used search_terms before search_documents for advisory queries"
    )
    tool_sequencing: SubDimensionScore = Field(
        ..., 
        description="Correct tool order (geocode→service, scheme_codes→scheme_info)"
    )
    search_quality: SubDimensionScore = Field(
        ..., 
        description="Effective 2-5 word English queries with relevant terms"
    )
    output_hygiene: SubDimensionScore = Field(
        ..., 
        description="No tool names leaked, no internal thinking exposed, no artifacts"
    )


# =============================================================================
# DIMENSION 2: FACTUAL GROUNDING
# =============================================================================

class FactualGrounding(BaseModel):
    """Evaluates whether claims are supported by tool outputs with no fabrication or harm."""
    
    source_alignment: SubDimensionScore = Field(
        ..., 
        description="All factual claims traceable to specific tool outputs"
    )
    no_fabrication: SubDimensionScore = Field(
        ..., 
        description="No invented data when tools return empty; gaps acknowledged"
    )
    citation_accuracy: SubDimensionScore = Field(
        ..., 
        description="Sources cited with farmer-friendly names, not internal tool names"
    )
    safety_compliance: SubDimensionScore = Field(
        ..., 
        description="Correct dosages, no banned chemicals, legal practices only"
    )


# =============================================================================
# DIMENSION 3: RESPONSE USEFULNESS
# =============================================================================

class ResponseUsefulness(BaseModel):
    """Evaluates whether the response is actually helpful to the farmer."""
    
    completeness: SubDimensionScore = Field(
        ..., 
        description="Addresses all parts of farmer's query"
    )
    actionability: SubDimensionScore = Field(
        ..., 
        description="Specific, timed, quantified advice (varieties, dosages, timing)"
    )
    context_fit: SubDimensionScore = Field(
        ..., 
        description="Uses farmer profile/location data appropriately when available"
    )
    clarity: SubDimensionScore = Field(
        ..., 
        description="Clear structure, readable formatting, appropriate length"
    )
    conversation_closure: SubDimensionScore = Field(
        ..., 
        description="Ends with relevant, specific follow-up question or next step"
    )


# =============================================================================
# DIMENSION 4: MARATHI LINGUISTIC QUALITY
# =============================================================================

class MarathiQuality(BaseModel):
    """Evaluates Marathi language correctness, terminology, and naturalness."""
    
    grammar: SubDimensionScore = Field(
        ..., 
        description="Grammatically correct, complete sentences"
    )
    terminology: SubDimensionScore = Field(
        ..., 
        description="Uses proper Marathi agricultural terms from glossary"
    )
    language_purity: SubDimensionScore = Field(
        ..., 
        description="No inappropriate English-Marathi mixing within sentences"
    )
    fluency: SubDimensionScore = Field(
        ..., 
        description="Reads naturally, conversational tone for rural context"
    )


# =============================================================================
# MAIN EVALUATION RESULT
# =============================================================================

class EvaluationResult(BaseModel):
    """
    Complete evaluation result for a MahaVistaar agricultural assistant response.
    
    Evaluates across 4 dimensions (16 sub-dimensions total):
    - Process Fidelity: Workflow compliance and output hygiene
    - Factual Grounding: Source alignment, no fabrication, safety
    - Response Usefulness: Completeness, actionability, clarity
    - Marathi Quality: Grammar, terminology, fluency
    
    Critical failures (score=1) in safety_compliance, no_fabrication, or 
    source_alignment result in overall_pass=False regardless of other scores.
    """
    
    process_fidelity: ProcessFidelity
    factual_grounding: FactualGrounding
    response_usefulness: ResponseUsefulness
    marathi_quality: MarathiQuality
    
    summary: str = Field(
        ..., 
        description="2-3 sentences: key strength + top improvement area"
    )

    def _calculate_dimension_average(self, dimension: BaseModel) -> Optional[float]:
        """Calculate average score for a dimension, excluding N/A (None) scores."""
        scores = []
        for field_name in dimension.model_fields:
            sub_dim: SubDimensionScore = getattr(dimension, field_name)
            if sub_dim.score is not None:
                scores.append(sub_dim.score.value)
        return round(sum(scores) / len(scores), 2) if scores else None

    def _get_critical_failures(self) -> List[str]:
        """Identify critical failures (score=1) in safety-critical sub-dimensions."""
        critical_fields = [
            ("factual_grounding", "safety_compliance"),
            ("factual_grounding", "no_fabrication"),
            ("factual_grounding", "source_alignment"),
        ]
        failures = []
        for dim_name, field_name in critical_fields:
            dimension = getattr(self, dim_name)
            sub_dim: SubDimensionScore = getattr(dimension, field_name)
            if sub_dim.score == Rating.UNACCEPTABLE:
                failures.append(f"{dim_name}.{field_name}")
        return failures

    def _sub_dim_to_dict(self, sub_dim: SubDimensionScore) -> Dict[str, Any]:
        """Convert sub-dimension to dict with numeric score."""
        return {
            "score": sub_dim.score.value if sub_dim.score else None,
            "evidence": sub_dim.evidence
        }

    def _dimension_to_dict(self, dimension: BaseModel) -> Dict[str, Any]:
        """Convert dimension to dict with numeric scores."""
        result = {}
        for field_name in dimension.model_fields:
            sub_dim: SubDimensionScore = getattr(dimension, field_name)
            result[field_name] = self._sub_dim_to_dict(sub_dim)
        return result

    def to_eval_dict(self) -> Dict[str, Any]:
        """
        Export evaluation as JSON-serializable dict with computed metrics.
        """
        dimensions = {
            "process_fidelity": {
                "scores": self._dimension_to_dict(self.process_fidelity),
                "average": self._calculate_dimension_average(self.process_fidelity)
            },
            "factual_grounding": {
                "scores": self._dimension_to_dict(self.factual_grounding),
                "average": self._calculate_dimension_average(self.factual_grounding)
            },
            "response_usefulness": {
                "scores": self._dimension_to_dict(self.response_usefulness),
                "average": self._calculate_dimension_average(self.response_usefulness)
            },
            "marathi_quality": {
                "scores": self._dimension_to_dict(self.marathi_quality),
                "average": self._calculate_dimension_average(self.marathi_quality)
            }
        }
        
        dim_averages = [d["average"] for d in dimensions.values() if d["average"] is not None]
        overall_average = round(sum(dim_averages) / len(dim_averages), 2) if dim_averages else None
        
        critical_failures = self._get_critical_failures()
        
        return {
            "dimensions": dimensions,
            "summary": self.summary,
            "metrics": {
                "overall_average": overall_average,
                "critical_failures": critical_failures,
                "critical_failure_count": len(critical_failures),
                "overall_pass": len(critical_failures) == 0
            }
        }

In [5]:
models   = [
            'Qwen_Qwen3-14b', 
           "kenpath_mhv_vistaar_qwen3-14b_v0.1",
            'gpt-4.1', 'gpt-4.1-mini',
           "kenpath_mhv_vistaar_qwen3-14b_v0.2",          
         'mistralai_Mistral-Small-3.2-24B-Instruct-2506',
    'openai_gpt-oss-20b',
           ]

for model_name in models:
    model_data_fp = os.path.join(model_dir,model_name,'data.json')
    data = json.load(open(model_data_fp))
    print(len(data))

470
470
470
470
470
470
470


In [6]:
from pyperclip import copy
import json

In [12]:
from helpers.utils import get_prompt


def format_agent_record(record):
    """
    Formats a Type-2 'Agent Record' (with separated question/answer/turns)
    into the same clean transcript format.
    Includes 'thinking' traces.
    """
    lines = []
    
    # --- 1. Context Header ---
    cat = record.get('category', 'Unknown')
    
    # Map 'agristack_required' to Icon
    agristack_val = record.get('agristack_required', 'No')
    agristack_icon = "✅" if agristack_val == 'Yes' else "❌"
    
    lines.append(f"🏷️  Context: [{cat}] | Agristack: {agristack_icon}")
    lines.append("=" * 60)
    lines.append("")

    # --- 2. USER QUERY ---
    lines.append("👤 USER:")
    lines.append(f"{record.get('question', '').strip()}")
    lines.append("-" * 40)

    # --- 3. MIDDLE STEPS (Agent Turns) ---
    for turn in record.get('agent_turns', []):
        for part in turn.get('parts', []):
            part_kind = part.get('part_kind')
            
            # A. THINKING / REASONING
            if part_kind == 'thinking':
                content = part.get('content', '')
                lines.append("💭 THOUGHT:")
                lines.append(f"{str(content).strip()}")
                lines.append("-" * 40)

            # B. TOOL CALL
            elif part_kind == 'tool-call':
                name = part.get('tool_name')
                args = part.get('args', '{}')
                
                lines.append(f"⚙️  ACTION: {name}")
                lines.append(f"    Args: {args}")

            # C. TOOL RETURN
            elif part_kind == 'tool-return':
                content = part.get('content', '')
                
                # formatting check: is it json?
                if isinstance(content, (dict, list)):
                     content_str = json.dumps(content, ensure_ascii=False, indent=2)
                else:
                     content_str = str(content)

                lines.append(f"📥 RESULT: {content_str.strip()}")
                lines.append("-" * 40)

    # --- 4. FINAL ANSWER ---
    lines.append("🤖 ASSISTANT:")
    lines.append(f"{record.get('answer', '').strip()}")
    lines.append("=" * 60)

    return "\n".join(lines)

In [ ]:
model_name  = random.choice(models)
model_data_fp = os.path.join(model_dir,model_name,'data.json')
data = json.load(open(model_data_fp))

print(model_name)
record = random.choice(data)

In [ ]:
set([r['category'] for r in data])

In [ ]:
from evaluation.evaluator import evaluation_agent, format_agent_record, EvaluationDeps

In [ ]:
message  = format_agent_record(record)
category = record.get("category", "")

In [ ]:
from pyperclip import copy

In [ ]:
eval_result = await evaluation_agent.run(
    message,
    deps=EvaluationDeps(category=category)
)

In [ ]:
copy(message)

In [ ]:
eval_result.output.to_eval_dict()

In [ ]:

'Scheme',
'MahaDBT',
# 'Mandi Price',

'Agri Assistant Contact',


# 'CHC',
# 'KVK',
# 'Soil Lab',
# 'Warehouse',

# 'Weather Forecast',
# 'Weather Historical'

In [ ]:
from pyperclip import copy

In [ ]:
import json
import pandas as pd
import plotly.graph_objects as go
import numpy as np


In [ ]:
df